# Case 3: Retinitis pigmentosa

*Hands-On Electroretinography in the Age of AI*, Chapter 19, §19.3.

Ms. F. is in the ≤35y age stratum. Her recording passes the signal quality check (DTL, SNR 6.3 for DA 3.0 and 9.1 for LA 30 Hz). Her DA 0.01 response lies below the noise floor, so its b-wave is not recordable. Chapter 11 §11.1 codes such a physiological absence with a sentinel value, and the pipeline assigns RED to it. This notebook assigns the zones of Table 19.4 with the pipeline code and checks the zones and printed limits against the book.

The values are constructed for teaching; they are not patient data. Limits: Baker et al. (2025), silver thread, ≤35y.

**Setup.** In Google Colab, the first cell clones the repository. Elsewhere, run the notebook from the repository.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('/content/ERG-Analysis-API'):
    !git clone -q https://github.com/AI-Fairness-com/ERG-Analysis-API.git /content/ERG-Analysis-API
sys.path.insert(0, '/content/ERG-Analysis-API/examples')
sys.path.insert(0, os.path.abspath('examples') if os.path.isdir('examples') else os.getcwd())
from case_helpers import zone_table, check, REPORT

**Table 19.4 values.** A not-recordable wave is entered with the Chapter 11 sentinel: −9999 for an amplitude and 9999 for an implicit time.

In [2]:
ABSENT_AMP, ABSENT_TIME = -9999.0, 9999.0   # Chapter 11 sentinels
ROWS = [
    ('DA 0.01', 'b-wave amplitude',      ABSENT_AMP,  'RED',   (138, 297)),
    ('DA 0.01', 'b-wave implicit time',  ABSENT_TIME, 'RED',   (72, 99.5)),
    ('DA 3.0',  'a-wave amplitude',      32,   'RED',   (119, 238)),
    ('DA 3.0',  'a-wave implicit time',  19.5, 'RED',   (13.5, 16)),
    ('DA 3.0',  'b-wave amplitude',      41,   'RED',   (191, 404)),
    ('DA 3.0',  'b-wave implicit time',  72,   'RED',   (45.5, 58.5)),
    ('DA 10.0', 'a-wave amplitude',      50,   'RED',   (136, 274)),
    ('DA 10.0', 'a-wave implicit time',  14.8, 'RED',   (10, 14.5)),
    ('DA 10.0', 'b-wave amplitude',      60,   'RED',   (202, 409)),
    ('DA 10.0', 'b-wave implicit time',  58,   'GREEN', (47, 59.5)),
    ('LA 3.0',  'a-wave amplitude',      14,   'GREEN', (12, 43)),
    ('LA 3.0',  'a-wave implicit time',  14.8, 'GREEN', (13, 15.5)),
    ('LA 3.0',  'b-wave amplitude',      70,   'AMBER', (73, 227)),
    ('LA 3.0',  'b-wave implicit time',  29.8, 'GREEN', (27, 31.5)),
    ('LA 30 Hz', 'flicker amplitude',    46,   'AMBER', (50, 166)),
    ('LA 30 Hz', 'flicker implicit time', 27.5, 'GREEN', (24, 28)),
    ('DA 3.0',  'b/a ratio',             1.3,  'Not zoned'),
    ('DA 3.0',  'OP2 amplitude',         3,    'Not zoned'),
    ('LA 3.0',  'PhNR amplitude',        26.1, 'Not zoned'),
]

**Zones.** The dark-adapted (rod-driven) responses are RED, while the cone responses are GREEN or AMBER (§19.3.2).

In [3]:
table, signal = zone_table(ROWS, age=28)
table['Value'] = table['Value'].replace({ABSENT_AMP: 'Not recordable',
                                         ABSENT_TIME: 'Not recordable'})
print(table.drop(columns='Match').to_string(index=False))
print('\nComposite Traffic Light:', signal)
check(table)

Protocol             Parameter          Value                            Limits Zone (code) Zone (book)
 DA 0.01      b-wave amplitude Not recordable   138 to 297 (L_lo 130, U_hi 319)         RED         RED
 DA 0.01  b-wave implicit time Not recordable  72 to 99.5 (L_lo 66.5, U_hi 106)         RED         RED
  DA 3.0      a-wave amplitude           32.0   119 to 238 (L_lo 109, U_hi 261)         RED         RED
  DA 3.0  a-wave implicit time           19.5     13.5 to 16 (L_lo 13, U_hi 17)         RED         RED
  DA 3.0      b-wave amplitude           41.0   191 to 404 (L_lo 178, U_hi 421)         RED         RED
  DA 3.0  b-wave implicit time           72.0 45.5 to 58.5 (L_lo 42.5, U_hi 60)         RED         RED
 DA 10.0      a-wave amplitude           50.0   136 to 274 (L_lo 130, U_hi 283)         RED         RED
 DA 10.0  a-wave implicit time           14.8   10 to 14.5 (L_lo 10, U_hi 14.5)         RED         RED
 DA 10.0      b-wave amplitude           60.0   202 to 409 (L_lo

**Rod and cone responses compared.** Counting the zones by light adaptation shows the rod-predominant pattern of retinitis pigmentosa.

In [4]:
import pandas as pd
table['Adaptation'] = table['Protocol'].str[:2].map({'DA': 'Dark-adapted', 'LA': 'Light-adapted'})
zoned = table[table['Zone (code)'] != 'NOT ZONED']
print(pd.crosstab(zoned['Adaptation'], zoned['Zone (code)']))

Zone (code)    AMBER  GREEN  RED
Adaptation                      
Dark-adapted       0      1    9
Light-adapted      2      4    0


**Saliency regions (§19.3.3).** The region scores of the constructed Level 2 saliency map are listed for reference. The b-wave and OP burst regions, where the signal is missing, carry the weight; the pre-stimulus interval stays below the 0.5 noise-warning level.

In [5]:
regions = {'b_wave_band': 1.00, 'op_burst_band': 0.62,
           'phnr_band': 0.05, 'prestim_noise': 0.02}
for name, score in regions.items():
    print(f'{name:14s} {score:.2f}')
print('Noise warning:', 'YES' if regions['prestim_noise'] >= 0.5 else 'no')

b_wave_band    1.00
op_burst_band  0.62
phnr_band      0.05
prestim_noise  0.02
Noise warning: no
